# Barra prototype — 220-name stratified sample

Builds the inputs for a USE4-style risk model on a deliberately small universe:
**20 securities per GICS sector × 11 sectors = 220**, drawn once from the
securities alive at **2005-12-31** and followed forward.

Run top to bottom. Every stage leaves its DataFrame in memory and prints a
sanity check, so you can stop anywhere and poke at the intermediate result.

### Running it more than once

Run it top to bottom every time — **do not start from a middle cell.** The two
flags in the *Run mode* cell decide what actually gets recomputed, and by default
a second run reads the roster and panel straight back out of TimescaleDB without
touching WRDS at all. Stage 3 always re-runs: it is pure pandas, it is cheap, and
it is the stage you are most likely iterating on.

| Want to… | `REDRAW_UNIVERSE` | `RELOAD_PANEL` | Hits WRDS |
|---|---|---|---|
| Re-standardize, change descriptors, tune winsorization | `False` | `False` | no |
| Pull newer months for the same 220 names | `False` | `True` | yes |
| Draw a different sample | `True` | `True` | yes |

⚠️ **`REDRAW_UNIVERSE = True` changes which 220 companies you are modelling, and
the fixed seed does not protect you.** The draw is reproducible only for a fixed
sampling frame, and that frame is built from live WRDS data — next month the
return-coverage counts are larger, so different securities clear the
`MIN_RETURN_MONTHS` screen, the eligible pool changes size, and the same seed
selects different companies. Anything computed against the old roster stops being
comparable. Redraw deliberately, not by habit.

### Why a sample at all

The full-universe pull produced three months of data with returns in only one of
them, which makes the factor covariance matrix **F** rank-1 and the model
unbuildable. A small universe with a *long* history is the opposite trade, and it
is the right one: `F` needs `T >> K`, not large `N`.

### Why stratified rather than uniform

Utilities is ~3% of the US listed universe. A uniform 220-name draw lands ≤1
Utilities name about 1.4% of the time, and ≤2 about 6% of the time. An industry
dummy with a single member fits that asset's return exactly, so its residual —
and therefore its specific variance — is forced to zero. That puts a zero on the
diagonal of `Δ`, makes `Σ` singular, and tells the optimizer that a portfolio
concentrated in that one stock is riskless. Stratifying removes the failure mode
by construction.

### Stages

| Stage | Output table | What it does |
|---|---|---|
| 1 | `ryan_sample_universe` | Build the eligible frame, draw the roster |
| 2 | `ryan_sample_panel` | Pull full monthly history, attach identifiers date-aware |
| 3 | `ryan_sample_exposures` | Winsorize, cap-weighted standardize, industry dummies |

### Prerequisites

```bash
cp .env.template .env    # fill in WRDS_USERNAME / WRDS_PASSWORD
docker compose up -d
```

Then open this notebook at http://localhost:8888 (token `qtab_jupyter_2024`), or
run it against the published port from the host — `db.py` resolves the hostname
either way.

In [ ]:
import os
import sys

# The helper modules sit next to this notebook. Jupyter starts with cwd = the
# notebook's directory, so this works in the container and on the host alike.
sys.path.insert(0, os.getcwd())

# Pick up edits to the .py modules without restarting the kernel. The logic
# lives in those modules, so without this a one-line fix to wrds_io or
# standardize means losing every loaded DataFrame to a kernel restart.
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import config
import db
import gics
import sampling
import standardize

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

print(f"anchor              {config.ANCHOR_EOM}")
print(f"names per sector    {config.NAMES_PER_SECTOR}  (→ {config.NAMES_PER_SECTOR * 11} target)")
print(f"min return months   {config.MIN_RETURN_MONTHS}")
print(f"seed                {config.RANDOM_SEED}")
print(f"descriptors         {', '.join(config.all_descriptors())}")
print(f"dependent variable  {config.RETURN_COL}  (never standardized)")

In [ ]:
engine = db.get_engine()
db.check_connection(engine)

---
## Run mode

On a first run both tables are absent, so everything builds. On later runs these
two flags decide what gets recomputed.

In [ ]:
# Set these, then run the whole notebook top to bottom.
REDRAW_UNIVERSE = False   # True → re-draw the roster. CHANGES WHICH 220 COMPANIES.
RELOAD_PANEL = False      # True → re-pull monthly history for the existing roster.

have_universe = db.table_exists(engine, config.TBL_UNIVERSE)
have_panel = db.table_exists(engine, config.TBL_PANEL)

# A fresh universe invalidates the stored panel, so rebuilding one forces the other.
build_universe = REDRAW_UNIVERSE or not have_universe
build_panel = RELOAD_PANEL or not have_panel or build_universe
need_wrds = build_universe or build_panel


def skip(reason="already in TimescaleDB"):
    """Marker for a cell whose work was resumed rather than recomputed."""
    print(f"⏭  skipped — {reason}")


print(f"{config.TBL_UNIVERSE:<26} {'found' if have_universe else 'absent':<7} → "
      f"{'REBUILD' if build_universe else 'resume from database'}")
print(f"{config.TBL_PANEL:<26} {'found' if have_panel else 'absent':<7} → "
      f"{'REBUILD' if build_panel else 'resume from database'}")
print(f"{'stage 3 exposures':<26} {'':<7} → ALWAYS recomputed (pure pandas, cheap)")
print(f"\nWRDS connection needed: {'yes' if need_wrds else 'no'}")

if build_universe and have_universe:
    print("\n⚠️  REDRAW_UNIVERSE replaces the existing roster with a different set of\n"
          "    companies — the fixed seed does not prevent this, because the eligible\n"
          "    frame shifts as WRDS accumulates data. Results computed against the old\n"
          "    roster will not be comparable.")

---
## Stage 1 — Build the sampling frame and draw the roster

Three ingredients: who was alive at the anchor, who has enough return history to
support a specific-risk estimate, and what sector each one is in.

In [ ]:
# Imported here rather than at the top so the notebook can be opened and the
# modules inspected without the wrds package installed; wrds_io imports it lazily.
if need_wrds:
    import wrds_io

    conn = wrds_io.connect()
else:
    conn = None
    skip("no WRDS work required this run")

In [ ]:
if build_universe:
    # Everything in the JKP US universe alive at the anchor date.
    # Note we key on `id` (JKP's security identifier), not ticker: tickers get
    # reused across firms — the old loader found BCS mapping to both LIFESTYLE
    # RESTAURANT and BARCLAYS PLC — change over time, and are missing for a large
    # share of crsp.stocknames.
    anchor = wrds_io.fetch_anchor_cross_section(conn)
    display(anchor.head())
else:
    skip()

In [ ]:
if build_universe:
    # How many months of *returns* each security has from the anchor forward.
    # Counting ret_exc rather than rows is the point: a month with no return cannot
    # enter the cross-sectional regression. This also screens out the Compustat-only
    # rows that carry no CRSP link and therefore no return at all — the defect that
    # left the full-universe pull with returns in one month out of three.
    coverage = wrds_io.fetch_return_coverage(conn)
    display(coverage.describe())
else:
    skip()

In [ ]:
if build_universe:
    fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
    ax[0].hist(coverage["n_ret"].dropna(), bins=60, color="#4C78A8")
    ax[0].axvline(config.MIN_RETURN_MONTHS, color="#E45756", ls="--",
                  label=f"min = {config.MIN_RETURN_MONTHS}")
    ax[0].set(title="Months with a return, per security", xlabel="n_ret", ylabel="securities")
    ax[0].legend()

    ax[1].scatter(coverage["n_months"], coverage["n_ret"], s=3, alpha=.25, color="#4C78A8")
    ax[1].plot([0, coverage["n_months"].max()], [0, coverage["n_months"].max()],
               color="#E45756", ls="--", lw=1, label="all months have a return")
    ax[1].set(title="Return coverage vs months present", xlabel="n_months", ylabel="n_ret")
    ax[1].legend()
    plt.tight_layout()
    plt.show()

    gap = coverage["n_months"] - coverage["n_ret"]
    print(f"securities where every month has a return: {(gap == 0).mean():.1%}")
    print(f"median missing-return months:              {gap.median():.0f}")
else:
    skip()

In [ ]:
if need_wrds:
    # GICS from comp.company, with names resolved from the hardcoded tables in
    # gics.py rather than the Kaggle map — which mislabelled American Airlines
    # (gsector 20, Industrials) as "Materials" and missed ~6.6% of rows outright.
    # Needed by stage 2 as well as stage 1, hence need_wrds rather than build_universe.
    gics_df = wrds_io.fetch_gics(conn)
    display(gics_df.head())
else:
    skip("sector labels already stored on the panel")

In [ ]:
if build_universe:
    frame = sampling.build_sampling_frame(anchor, coverage, gics_df)
else:
    skip()

### Sanity check the frame before drawing from it

If any of these look wrong, the roster will be wrong too and every later stage
inherits it.

In [ ]:
if build_universe:
    checks = {
        "one row per id": not frame["id"].duplicated().any(),
        "every row has a sector": frame["sector_code"].notna().all(),
        "every sector name resolved": frame["sector_name"].notna().all(),
        "all 11 sectors represented": frame["sector_code"].nunique() == 11,
        f"every row has >= {config.MIN_RETURN_MONTHS} return months":
            (frame["n_ret"] >= config.MIN_RETURN_MONTHS).all(),
        "enough names to fill every sector":
            frame.groupby("sector_code").size().min() >= config.NAMES_PER_SECTOR,
    }
    for name, ok in checks.items():
        print(f"{'✅' if ok else '❌'} {name}")

    thin = frame.groupby(["sector_code", "sector_name"]).size().sort_values()
    print(f"\nthinnest sectors in the eligible frame:\n{thin.head(4).to_string()}")
else:
    skip()

In [ ]:
if build_universe:
    roster = sampling.draw_stratified_sample(frame)
    db.write_table(roster, config.TBL_UNIVERSE, engine, unique_on=["id"])
else:
    roster = db.read_table(config.TBL_UNIVERSE, engine)
    print(f"📂 loaded roster of {len(roster):,} names from {config.TBL_UNIVERSE}")
    print(f"   drawn with seed {roster['sample_seed'].iloc[0]}, "
          f"anchored at {pd.to_datetime(roster['anchor_eom'].iloc[0]).date()}")
    print(roster.groupby("sector_name").size().to_string())

roster[["id", "gvkey", "sector_code", "sector_name", "conm", "n_ret"]].head(15)

---
## Stage 2 — Load the panel and attach identifiers

The window is anchored on an explicit date, not `datetime.now() - months_back*30`.
That arithmetic is what silently produced a 3-month panel: run in Dec 2025 with
`months_back=12` it asked for `eom >= 2024-12-10`, and the table ends 2025-02-28.

In [ ]:
if build_panel:
    ids = roster["id"].tolist()
    panel = wrds_io.fetch_panel(conn, ids)
else:
    panel = db.read_table(config.TBL_PANEL, engine)
    panel["eom"] = pd.to_datetime(panel["eom"])
    print(f"📂 loaded {len(panel):,} rows / {panel['id'].nunique():,} securities "
          f"from {config.TBL_PANEL}")
    print(f"   {panel['eom'].min().date()} → {panel['eom'].max().date()}")

panel.head()

In [ ]:
if build_panel:
    # Date-aware identifier joins.
    #
    # The old code merged the link table on gvkey alone, which fanned 14,813 rows
    # out to 47,448 because a gvkey that mapped to permno X in 1990 and Y in 2010
    # contributed both rows at every date. Restricting to the link whose
    # [linkdt, linkenddt] window contains eom gives exactly one answer per row, so
    # there is no drop_duplicates cleanup step to get wrong.
    links = wrds_io.fetch_ccm_links(conn)
    panel = sampling.attach_permno(panel, links)

    permnos = panel["permno_ccm"].dropna().unique().tolist()
    names = wrds_io.fetch_ticker_history(conn, permnos)
    panel = sampling.attach_ticker(panel, names)

    # Sector, for the industry dummies. Codes carry the model; names are labels.
    panel = panel.merge(gics_df, on="gvkey", how="left")
    panel = gics.attach_names(panel)
    display(panel.head())
else:
    skip("identifiers already resolved in the stored panel")

### Sanity check the panel

The row-count check is the important one — it is exactly what the old pipeline
got wrong and then papered over with two `drop_duplicates` calls.

In [ ]:
n_expected = panel[["id", "eom"]].drop_duplicates().shape[0]
checks = {
    "(id, eom) is unique — no join fan-out": not panel.duplicated(subset=["id", "eom"]).any(),
    "row count equals distinct (id, eom)": len(panel) == n_expected,
    "every row has a sector code": panel["sector_code"].notna().all(),
    "all 11 sectors still present": panel["sector_code"].nunique() == 11,
    "roster fully represented": panel["id"].nunique() == len(roster),
    "returns present in most rows": panel[config.RETURN_COL].notna().mean() > 0.8,
}
for name, ok in checks.items():
    print(f"{'✅' if ok else '❌'} {name}")

print(f"\nrows              {len(panel):,}")
print(f"securities        {panel['id'].nunique():,} of {len(roster):,} drawn")
print(f"months            {panel['eom'].nunique():,}  ({panel['eom'].min().date()} → {panel['eom'].max().date()})")
print(f"return coverage   {panel[config.RETURN_COL].notna().mean():.1%}")

In [ ]:
# Per-month coverage. Falling asset counts are expected and wanted -- this is a
# fixed 2005 cohort, so the decline *is* the delistings we deliberately kept.
per_month = panel.groupby("eom").agg(
    n_assets=("id", "nunique"),
    n_returns=(config.RETURN_COL, lambda s: s.notna().sum()),
    n_sectors=("sector_code", "nunique"),
)

fig, ax = plt.subplots(1, 2, figsize=(13, 3.5))
ax[0].plot(per_month.index, per_month["n_assets"], label="assets", color="#4C78A8")
ax[0].plot(per_month.index, per_month["n_returns"], label="with a return", color="#E45756")
ax[0].set(title="Assets per cross-section", ylabel="count")
ax[0].legend()
ax[1].plot(per_month.index, per_month["n_sectors"], color="#54A24B")
ax[1].axhline(11, color="grey", ls=":", lw=1)
ax[1].set(title="Sectors represented per cross-section", ylim=(0, 12))
plt.tight_layout()
plt.show()

per_month.describe()

In [ ]:
# Null report. Expect single-digit percentages on the fundamentals (financials
# lack gp_at, negative book equity kills be_me, young firms lack lagged data).
# A characteristic above ~30% here is a problem worth chasing, not noise.
nulls = (panel.isna().mean() * 100).sort_values(ascending=False)
nulls[nulls > 0].round(2).to_frame("% missing")

In [ ]:
if build_panel:
    db.write_table(panel, config.TBL_PANEL, engine, unique_on=["id", "eom"])
else:
    skip("panel unchanged, nothing to write")

---
## Stage 3 — Standardize exposures and build industry dummies

Always recomputed, whatever the run mode: it is pure pandas over a panel already
in memory, and it is the stage you are most likely changing.

Three differences from a plain z-score, each of which changes the numbers:

- **Robust winsorization first.** Trimming at 3σ of the *ordinary* standard
  deviation does not work, because the outlier being removed is itself inflating
  that standard deviation. Median and MAD are immune to it.
- **Cap-weighted mean, equal-weighted std.** USE4 sets the *cap-weighted* mean to
  zero, which is what makes the market portfolio style-neutral. A plain z-score
  zeroes the equal-weighted mean instead, leaving the market factor and the style
  factors overlapping.
- **log(market cap) for size.** Raw market cap is skewed enough that z-scoring it
  puts a handful of mega-caps at +40σ.

And `ret_exc` is never standardized — it is the dependent variable, not an
exposure.

In [ ]:
exposures = standardize.standardize_panel(panel)
print("\ncombining collinear descriptors into one factor each:")
exposures = standardize.combine_descriptors(exposures)
exposures, dummy_cols = standardize.build_industry_dummies(exposures)

factor_cols = [c for c in exposures.columns if c.startswith("f_")]
print(f"\nstyle factors:    {', '.join(factor_cols)}")
print(f"industry dummies: {len(dummy_cols)}")

### Verify the standardization actually holds

These are the properties the regression depends on, so check them rather than
assume them.

In [ ]:
w = pd.to_numeric(exposures[config.CAP_COL], errors="coerce")
rows = []
for col in factor_cols:
    x, g = exposures[col], exposures["eom"]
    ok = x.notna() & w.notna() & (w > 0)
    cwm = (
        pd.DataFrame({"xw": (x * w).where(ok), "w": w.where(ok), "g": g})
        .groupby("g").sum(min_count=1).pipe(lambda d: d["xw"] / d["w"])
    )
    rows.append({
        "factor": col,
        "max |cap-wtd mean|": cwm.abs().max(),
        "max |sd - 1|": (x.groupby(g).std() - 1).abs().max(),
        "max |z|": x.abs().max(),
        "% missing": x.isna().mean() * 100,
    })
summary = pd.DataFrame(rows).set_index("factor")
display(summary.round(4))

print(f"{'✅' if summary['max |cap-wtd mean|'].max() < 1e-8 else '❌'} cap-weighted mean is zero in every cross-section")
print(f"{'✅' if summary['max |sd - 1|'].max() < 1e-8 else '❌'} unit dispersion in every cross-section")
print(f"{'✅' if summary['max |z|'].max() < 6 else '❌'} no runaway exposures (max |z| = {summary['max |z|'].max():.2f})")

In [ ]:
# THE diagnostic for a small universe: how thin does the thinnest industry get?
# Any month where an industry drops to 1 member produces a zero specific variance
# for that asset and a singular Sigma. 2 is fragile. Watch this column.
report = standardize.coverage_report(exposures)
worst = report.nsmallest(8, "min_industry_n")

fig, ax = plt.subplots(figsize=(13, 3.2))
ax.plot(report["eom"], report["min_industry_n"], color="#4C78A8")
ax.axhline(3, color="#E45756", ls="--", lw=1, label="fragile below 3")
ax.set(title="Smallest industry membership, per cross-section", ylabel="assets in thinnest industry")
ax.legend()
plt.tight_layout()
plt.show()

print(f"minimum across all months: {report['min_industry_n'].min()}")
print(f"months with an industry below 3: {(report['min_industry_n'] < 3).sum()} of {len(report)}")
display(worst)

In [ ]:
# Collinearity between the combined style factors. Descriptor grouping already
# collapsed be_me/ni_me/ocf_me into one Value column; anything still above ~0.7
# here will make the individual factor returns unstable even where the fit is fine.
corr = exposures[factor_cols].corr()
fig, ax = plt.subplots(figsize=(6.5, 5.2))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(factor_cols)), [c[2:] for c in factor_cols], rotation=45, ha="right")
ax.set_yticks(range(len(factor_cols)), [c[2:] for c in factor_cols])
for i in range(len(factor_cols)):
    for j in range(len(factor_cols)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center",
                fontsize=8, color="white" if abs(corr.iloc[i, j]) > .55 else "black")
plt.colorbar(im, shrink=.8)
ax.set_title("Style factor correlations (pooled)")
plt.tight_layout()
plt.show()

off = corr.where(~np.eye(len(corr), dtype=bool)).abs().max().max()
print(f"{'✅' if off < 0.7 else '⚠️ '} largest off-diagonal |correlation| = {off:.3f}")

In [ ]:
keep = (
    ["id", "eom", "gvkey", "permno_ccm", "ticker", "conm",
     "sector_code", "sector_name", "group_code", "group_name",
     config.RETURN_COL, config.CAP_COL]
    + factor_cols
    + dummy_cols
)
out = exposures[[c for c in keep if c in exposures.columns]].copy()
db.write_table(out, config.TBL_EXPOSURES, engine, unique_on=["id", "eom"])
out.head()

---
## What this does not do yet

Stages 4 and 5 of the model are still open:

**Step 4 — factor and specific returns.** One cross-sectional WLS regression per
month, with weights ∝ √(market cap):

$$r_i = \sum_k X_{ik} f_k + u_i$$

Industry dummies and style factors go into the *same* regression. Note that the
design matrix `[market | industries | styles]` is rank-deficient by construction
— the industry dummies sum to the market's column of ones — so it needs USE4's
cap-weighted constraint $\sum_i w_i f_i^{\text{ind}} = 0$. Pre-orthogonalizing
the styles against the industry dummies does **not** fix this; the constraint is
required either way.

**Step 5 — the covariance matrix.**

$$\Sigma = X F X^\top + \Delta$$

`F` is the covariance of **all** factor returns jointly — one `K×K` matrix, not
separate industry and style matrices added together, which would drop every
industry–style cross-covariance. `Δ` is diagonal specific variance, Newey-West
adjusted for the serial correlation in each asset's specific-return series and
then Bayes-shrunk toward its size-decile mean.

With 220 assets you can build the full `220×220` `Σ` and run real bias tests —
compare $w^\top \Sigma w$ against realized out-of-sample portfolio variance. That
is the actual validation, and it works at this size.